# Deploy Workloads with Lakeflow Jobs

Training notebook covering Lakeflow Jobs: anatomy of a job, task types, multi-task workflows, scheduling, compute options, monitoring, and retry handling.

## 📋 What We'll Cover

1. **What are Lakeflow Jobs** — purpose and when to use jobs vs notebooks vs pipelines
2. **Anatomy of a Job** — tasks, clusters, triggers, and run lifecycle
3. **Task Types** — notebook, Python script, JAR, pipeline, SQL, dbt, and more
4. **Multi-task Workflows & Dependencies** — chaining tasks with `depends_on`, fan-out/fan-in
5. **Scheduling** — cron, file arrival, table update, and continuous triggers
6. **Compute Options** — job clusters vs all-purpose clusters vs serverless
7. **Monitoring & Alerts** — run history, duration metrics, email/webhook notifications
8. **Retry & Error Handling** — retry policies, repair runs, `run_if` conditions
9. **SDK Example** — inspect and trigger a job programmatically

## 1. What are Lakeflow Jobs

Lakeflow Jobs is the **orchestration engine** of the Databricks platform. A Job is a scheduled or triggered unit of work that runs one or more tasks on managed compute.

### When to use a Job (vs just running a notebook)

| Scenario | Use |
|---|---|
| Run code on a schedule (nightly, hourly) | **Job** |
| Chain multiple steps with dependencies | **Job** |
| Notify on failure / retry on error | **Job** |
| Interactive exploration & development | Notebook (no job needed) |
| Continuously ingest streaming data | Lakeflow Spark Declarative Pipeline |

### Key concepts

| Term | Definition |
|---|---|
| **Job** | The container: name, tasks, schedule, alerts, permissions |
| **Run** | A single execution of the job (triggered or manual) |
| **Task** | One unit of work inside a job (notebook, script, pipeline…) |
| **Task cluster** | The compute on which a task runs |
| **Trigger** | What starts a run (schedule, API call, file arrival, etc.) |

## 2. Anatomy of a Job

```
 JOB: daily_sales_pipeline
 ├── Trigger      : cron  0 6 * * *  (daily at 06:00 UTC)
 ├── Tasks
 │   ├── [1] ingest_raw        Notebook  → job cluster A
 │   ├── [2] transform_silver  Notebook  → job cluster A  (depends on [1])
 │   ├── [3] load_gold         SQL file  → serverless      (depends on [2])
 │   └── [4] refresh_dash      Pipeline  → serverless      (depends on [3])
 ├── Notifications : email on failure/success
 └── Permissions   : CAN_MANAGE, CAN_VIEW, IS_OWNER
```

### Run lifecycle

```
Trigger fires
    ↓
Run created (PENDING)
    ↓
Cluster provisioned (RUNNING)
    ↓
Tasks execute in dependency order
    ↓
Run completes (SUCCEEDED / FAILED / CANCELED)
    ↓
Cluster terminates (job cluster) or stays up (all-purpose)
```

### Parallelism
Tasks with **no dependency relationship** run in parallel on separate clusters, e.g. tasks [2] and a separate data-quality check can both start once [1] completes.

## 3. Task Types

| Task Type | What it Runs | Typical Use |
|---|---|---|
| **Notebook** | A Databricks notebook (Python, SQL, Scala, R) | Data transformation, EDA, ML training |
| **Python script** | A `.py` file in a workspace path or Git repo | Modular ETL, library-based code |
| **Python wheel** | A packaged `.whl` deployed to DBFS or UC Volume | Versioned, reusable code |
| **JAR** | A compiled Scala/Java artifact | Legacy or performance-critical Spark jobs |
| **SQL** | A `.sql` file or Databricks SQL query | Data loading, aggregation, DDL |
| **Spark Declarative Pipeline** | A Lakeflow SDP pipeline | Bronze/silver/gold ingestion pipelines |
| **dbt** | A dbt project | SQL-first transformation layer |
| **Genie Code task** | An AI-driven automated task | Automated reporting, monitoring prompts |
| **Condition** | True/False branch based on previous task output | Dynamic workflow routing |
| **For each** | Iterate a task over a list of values | Parallel processing per entity |

> 💡 **Tip:** Use **Notebook** tasks during development (easy to debug), then convert to **Python script** tasks for production jobs (cleaner, version-controlled).

## 4. Multi-task Workflows & Dependencies

### Dependency patterns

**Sequential** (each task waits for the previous):
```
[A] ──► [B] ──► [C]
```

**Fan-out** (one task triggers many in parallel):
```
         ┌► [B]
[A] ────┤
         └► [C]
```

**Fan-in** (many tasks must complete before the next):
```
[A] ──►┐
       └► [C]
[B] ──►┘
```

**Diamond** (fan-out then fan-in):
```
         ┌► [B] ─►┐
[A] ────┤         └► [D]
         └► [C] ─►┘
```

### `run_if` conditions

| Condition | Meaning |
|---|---|
| `ALL_SUCCESS` | Run only if all upstream tasks succeeded (default) |
| `AT_LEAST_ONE_SUCCESS` | Run if at least one upstream task succeeded |
| `NONE_FAILED` | Run unless an upstream task explicitly failed |
| `ALL_DONE` | Run regardless of upstream outcome |
| `AT_LEAST_ONE_FAILED` | Run only if at least one task failed (e.g. alert task) |

### Task values (passing data between tasks)
A task can emit a string value:
```python
dbutils.jobs.taskValues.set(key="row_count", value=str(df.count()))
```
A downstream task reads it:
```python
row_count = dbutils.jobs.taskValues.get(taskKey="ingest_raw", key="row_count")
```

## 5. Scheduling

### Trigger types

| Trigger | Description | Example |
|---|---|---|
| **Scheduled (cron)** | Fixed interval using Quartz cron syntax | `0 0 6 * * ?` = daily at 06:00 |
| **File arrival** | Trigger when a new file lands in a UC Volume or cloud path | Monitor `/Volumes/raw/landing/` |
| **Table update** | Trigger when a specific Delta table is updated | `one_dbx_training.default.customer_batch` |
| **Continuous** | Re-trigger immediately after each run completes | Always-on processing |
| **Manual / API** | Trigger via UI button or REST API call | `POST /api/2.1/jobs/run-now` |

### Quartz cron cheat sheet

| Expression | Meaning |
|---|---|
| `0 0 6 * * ?` | Every day at 06:00 |
| `0 0/15 * * * ?` | Every 15 minutes |
| `0 0 8 ? * MON-FRI` | Weekdays at 08:00 |
| `0 0 0 1 * ?` | First day of every month at midnight |

> 💡 **Tip:** Use the **Cron expression builder** in the Jobs UI to avoid manual cron mistakes.

## 6. Compute Options

| Compute Type | Best For | Notes |
|---|---|---|
| **Serverless (recommended)** | Most notebook and SQL tasks | No startup delay; pay per second; auto-scales |
| **Job cluster** | Heavy Spark jobs needing specific configs | Provisioned fresh per run; terminates after |
| **All-purpose cluster** | Development tasks sharing a running cluster | Faster start; more expensive for long-idle jobs |
| **SQL warehouse** | SQL tasks, dbt tasks | Dedicated SQL optimized compute |

### Serverless advantages for Jobs

* **No cold-start delay** — tasks start in seconds vs minutes for job clusters
* **Per-task clusters** — each task can have its own serverless compute automatically
* **Cost** — billed only for active computation time

### When to use a Job cluster instead

* Spark configs that must be set at cluster creation (e.g. custom Spark properties, init scripts)
* Tasks requiring specific Databricks Runtime versions or pre-installed libraries
* Very long-running tasks where serverless time limits apply

## 7. Monitoring & Alerts

### Run history view
In the Jobs UI, every run shows:
* Start time, duration, trigger type
* Per-task status (success / failed / skipped)
* Cluster logs and driver output
* Spark UI link (for non-serverless)

### Key signals to monitor

| Signal | What it means |
|---|---|
| **Duration trend increasing** | Data volume growing, or a task is becoming inefficient |
| **Task failed** | Check driver logs and exception message |
| **Task skipped** | `run_if` condition not met; check upstream tasks |
| **Run queued for long** | Compute quota reached or cluster taking too long to start |

### Setting up notifications

1. In the Job editor → **Notifications** tab.
2. Add email addresses or webhook URLs.
3. Choose events: **On start**, **On success**, **On failure**, **Duration exceeded**.

> 💡 For critical production jobs, set a **Duration exceeded** alert at 2× the expected runtime as an early warning.

## 8. Retry & Error Handling

### Per-task retry policy

Each task can be configured with:
* **Max retries** — how many times to re-attempt on failure
* **Retry on timeout** — whether a timeout counts as a failure to retry
* **Min retry interval** — wait time before the next attempt

```
Task failed on attempt 1
    ↓ wait (retry interval)
Task retried on attempt 2
    ↓ if still fails
Task marked FAILED after max retries
```

### Repair runs

If a multi-task job fails midway, you do **not** need to re-run the whole job. Use **Repair run** to:
* Re-run only the failed task and all its downstream dependents
* Tasks that already succeeded are **not re-executed** (saves time and cost)

### Task-level timeouts

Set a maximum duration per task. If the task exceeds it, it is cancelled and the retry policy kicks in (if enabled).

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# --- List all jobs in the workspace ---
print(f"{'Job ID':<12} {'Name':<40} {'Created by':<30}")
print("─" * 82)
for job in w.jobs.list(expand_tasks=False):
    print(f"{job.job_id:<12} {(job.settings.name or 'unnamed')[:39]:<40} {str(job.creator_user_name)[:29]:<30}")

print("\n✅ To trigger a job run: w.jobs.run_now(job_id=<your_job_id>)")